<a href="https://colab.research.google.com/github/baxodirmamatvaliev-del/JaydariGPT/blob/main/Jaydari_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
%pip install transformers bitsandbytes datasets peft accelerate trl "torchao>=0.16.0"

In [22]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from trl import SFTTrainer

In [23]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" # DATA
tokenizer = AutoTokenizer.from_pretrained(model_id)


model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

print('Vocab size:', tokenizer.vocab_size)
print('Special tokens:', tokenizer.special_tokens_map)


# Quantization - MODUL xotirasini ozxolida kichikroq saqlash uchun
# m: 4K --> 1080p
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)



Vocab size: 32000
Special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}


In [24]:
bnb_config

model = AutoModelForCausalLM.from_pretrained( # llama ni data sini yuklab oldik
    model_id,  # llamani topish
    quantization_config=bnb_config,
    device_map="auto", # gpu / cpu
    dtype=torch.float16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [25]:
# Before Fine-Tuning
prompt = "Explain what a tokenizer is."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # GPU, CPU

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

print(model)
print("======================")
print(model.config)


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is.
LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
  

In [26]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")


Total parameters (including frozen 4-bit): 615,606,272


Datasets library
# Instruction tuning

In [27]:
from datasets import load_dataset

dataset = load_dataset('yahma/alpaca-cleaned', split='train')
dataset

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 51760
})

In [28]:
def generate_prompt(example):
  instruction = example['instruction']
  input_text = example['input']
  output_text = example['output']

  if input_text:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{input_text}\n\n"
        "### Response:\n"
        f"{output_text}\n\n"
    )
  else:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Response:\n"
        f"{output_text}\n\n"
    )

generate_prompt(dataset[1])

'### Instruction:\nWhat are the three primary colors?\n\n### Response:\nThe three primary colors are red, blue, and yellow. These colors are called primary because they cannot be created by mixing other colors and all other colors can be made by combining them in various proportions. In the additive color system, used for light, the primary colors are red, green, and blue (RGB).\n\n'

In [29]:
def formatting_func(example):
    return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)

In [30]:
dataset[0]['text'] #yangi future

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.\n\n'

In [31]:
dataset = dataset.select(range(7000)) #memoriylar tugap qolmasligi uchun 7000 data ni ozida instraction tuning qilamz

In [32]:
dataset = dataset.shuffle(seed=42) # endi random holatda 7000 malumotni ol / seed=42 malumotlarni ertaga ham bir xil saqlash

In [33]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})

In [34]:
# Full Fine-tuning  =>
# Cheap Fine-tuning =>
# PEFT => Parameter Efficient Fine Tuning
# OOM => Out of Memory

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,  # rank qancha kichik bolsa shuncha kichik parametrni tuning qiladi bu memoryga gpuga yaxshi
    lora_alpha=16,  #scaling qilib beradi , yani tozalash
    lora_dropout=0.05, #overfit bolishini oldini olish uchun
    task_type="CAUSAL_LM", # yengilloq task yoki qiyin task berishimiz mumkin
    target_modules=["q_proj", "v_proj"], #bular model attention qismida layeridan olinmoqda va bu ikkita layer eng muhim qilsmlari hisoblanadi
)

In [35]:
model = get_peft_model(model, lora_config)

In [36]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023
